This file simply shows the results of each default period chosen with different bar sizes for US stocks during normal trading hours
- Even if the number of items overflow in queries, technically the mechanism for downloading uses the earliest date fetched to construct the next query so we won't lose any data points. Instead there is a greater risk of having timeouts
- Conid `265598` is AAPL

In [1]:
import requests
from datetime import datetime, timezone
import urllib3
from zoneinfo import ZoneInfo
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import pandas as pd

pd.set_option("display.max_colwidth", None)

def parseIbkrTime(timeString: str) -> datetime:
    # Used for metadata returned
    return datetime.strptime(timeString, "%Y%m%d-%H:%M:%S").replace(tzinfo=timezone.utc)

def formatTime(time):
    if type(time) is str:
        time = parseIbkrTime(time)

    if type(time) is int:
        time = time / 1000.0
        time = datetime.fromtimestamp(time, tz=timezone.utc)

    if type(time) is datetime:
        if time.tzinfo is None:
            time = time.replace(tzinfo=timezone.utc)
        ny_time = time.astimezone(ZoneInfo("America/New_York"))
        fmt = "%Y-%m-%d %a %I:%M:%S %p"
        time = f"(utc) {time.strftime(fmt)}, (NY) {ny_time.strftime(fmt)}"
        
    return time

# The input date time is in utc and api expects utc I think
def dataQuery(
    conid: int,
    bar: str,
    startTime: datetime,
    period: str = None,
    outsideRth: bool = False
):
    startTime = startTime.strftime('%Y%m%d-%H:%M:%S')

    params = {
        'conid' : conid,
        'bar' : bar,
        'period' : period,
        'startTime' : startTime,
        'outsideRth' : outsideRth
    }
    r = requests.get("https://localhost:5000/v1/api/iserver/marketdata/history", params=params, verify=False)

    data = r.json()
    #print(data)
    formattedTime = formatTime(data['startTime'])

    print(f"{'startTime:':<20} {formattedTime}")
    print(f"{'chartPanStartTime:':<20} {formatTime(data['chartPanStartTime'])}")
    print(f"{'timePeriod:':<20} {data['timePeriod']}")
    print(f"{'barLength:':<20} {data['barLength']}")
    df = pd.DataFrame(data['data'])
    df['t'] = df['t'].apply(formatTime)
    return df


Note for bars equal and under 30 seconds, the date will need to be within 6 months of the current time  
Also note that the times returned are incorrect

In [5]:
dataQuery(
    conid = 265598,
    bar = '1S',
    period = '16min',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-09-08 Tue 07:43:59 PM, (NY) 2026-09-08 Tue 03:43:59 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          960s
barLength:           1


,o,c,h,l,v,t
0,316.14,316.13,316.14,316.13,8.925,"(utc) 2026-09-08 Tue 07:43:59 PM, (NY) 2026-09-08 Tue 03:43:59 PM"
1,316.14,316.12,316.14,316.11,121.500,"(utc) 2026-09-08 Tue 07:44:59 PM, (NY) 2026-09-08 Tue 03:44:59 PM"
2,316.12,316.12,316.12,316.11,8.200,"(utc) 2026-09-08 Tue 07:45:59 PM, (NY) 2026-09-08 Tue 03:45:59 PM"
3,316.12,316.14,316.14,316.12,19.000,"(utc) 2026-09-08 Tue 07:46:59 PM, (NY) 2026-09-08 Tue 03:46:59 PM"
4,316.13,316.14,316.14,316.13,2.000,"(utc) 2026-09-08 Tue 07:47:59 PM, (NY) 2026-09-08 Tue 03:47:59 PM"
...,...,...,...,...,...,...
955,316.45,316.41,316.47,316.40,173.975,"(utc) 2026-09-09 Wed 11:38:59 AM, (NY) 2026-09-09 Wed 07:38:59 AM"
956,316.41,316.41,316.45,316.38,320.000,"(utc) 2026-09-09 Wed 11:39:59 AM, (NY) 2026-09-09 Wed 07:39:59 AM"
957,316.42,316.35,316.42,316.35,304.850,"(utc) 2026-09-09 Wed 11:40:59 AM, (NY) 2026-09-09 Wed 07:40:59 AM"
958,316.35,316.40,316.43,316.34,1092.400,"(utc) 2026-09-09 Wed 11:41:59 AM, (NY) 2026-09-09 Wed 07:41:59 AM"


In [6]:
dataQuery(
    conid = 265598,
    bar = '5S',
    period = '80min',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-09-08 Tue 06:39:55 PM, (NY) 2026-09-08 Tue 02:39:55 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          4800s
barLength:           5


,o,c,h,l,v,t
0,315.83,315.86,315.88,315.83,239.725,"(utc) 2026-09-08 Tue 06:39:55 PM, (NY) 2026-09-08 Tue 02:39:55 PM"
1,315.85,315.80,315.85,315.80,57.900,"(utc) 2026-09-08 Tue 06:40:55 PM, (NY) 2026-09-08 Tue 02:40:55 PM"
2,315.81,315.81,315.81,315.80,1.475,"(utc) 2026-09-08 Tue 06:41:55 PM, (NY) 2026-09-08 Tue 02:41:55 PM"
3,315.81,315.77,315.81,315.77,52.825,"(utc) 2026-09-08 Tue 06:42:55 PM, (NY) 2026-09-08 Tue 02:42:55 PM"
4,315.77,315.81,315.81,315.77,57.350,"(utc) 2026-09-08 Tue 06:43:55 PM, (NY) 2026-09-08 Tue 02:43:55 PM"
...,...,...,...,...,...,...
955,315.99,316.12,316.16,315.99,1362.300,"(utc) 2026-09-09 Wed 10:34:55 AM, (NY) 2026-09-09 Wed 06:34:55 AM"
956,316.12,316.19,316.28,316.12,1717.075,"(utc) 2026-09-09 Wed 10:35:55 AM, (NY) 2026-09-09 Wed 06:35:55 AM"
957,316.20,316.34,316.38,316.20,1506.900,"(utc) 2026-09-09 Wed 10:36:55 AM, (NY) 2026-09-09 Wed 06:36:55 AM"
958,316.35,316.33,316.38,316.29,1911.375,"(utc) 2026-09-09 Wed 10:37:55 AM, (NY) 2026-09-09 Wed 06:37:55 AM"


In [7]:
dataQuery(
    conid = 265598,
    bar = '10S',
    period = '160min',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-09-08 Tue 05:19:50 PM, (NY) 2026-09-08 Tue 01:19:50 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          9600s
barLength:           10


,o,c,h,l,v,t
0,316.09,316.07,316.10,316.07,34.250,"(utc) 2026-09-08 Tue 05:19:50 PM, (NY) 2026-09-08 Tue 01:19:50 PM"
1,316.08,316.08,316.08,316.06,69.175,"(utc) 2026-09-08 Tue 05:20:50 PM, (NY) 2026-09-08 Tue 01:20:50 PM"
2,316.08,316.04,316.08,316.01,105.475,"(utc) 2026-09-08 Tue 05:21:50 PM, (NY) 2026-09-08 Tue 01:21:50 PM"
3,316.04,316.02,316.04,316.01,78.425,"(utc) 2026-09-08 Tue 05:22:50 PM, (NY) 2026-09-08 Tue 01:22:50 PM"
4,316.02,316.02,316.02,316.00,21.250,"(utc) 2026-09-08 Tue 05:23:50 PM, (NY) 2026-09-08 Tue 01:23:50 PM"
...,...,...,...,...,...,...
955,315.95,315.85,315.96,315.80,1765.975,"(utc) 2026-09-09 Wed 09:14:50 AM, (NY) 2026-09-09 Wed 05:14:50 AM"
956,315.85,315.94,315.95,315.82,1402.325,"(utc) 2026-09-09 Wed 09:15:50 AM, (NY) 2026-09-09 Wed 05:15:50 AM"
957,315.95,315.98,315.98,315.89,1689.075,"(utc) 2026-09-09 Wed 09:16:50 AM, (NY) 2026-09-09 Wed 05:16:50 AM"
958,315.99,316.19,316.28,315.99,3079.375,"(utc) 2026-09-09 Wed 09:17:50 AM, (NY) 2026-09-09 Wed 05:17:50 AM"


In [8]:
dataQuery(
    conid = 265598,
    bar = '15S',
    period = '240min',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-09-08 Tue 03:59:45 PM, (NY) 2026-09-08 Tue 11:59:45 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          14400s
barLength:           15


,o,c,h,l,v,t
0,316.15,316.08,316.15,316.06,251.475,"(utc) 2026-09-08 Tue 03:59:45 PM, (NY) 2026-09-08 Tue 11:59:45 AM"
1,316.06,316.03,316.13,316.00,429.875,"(utc) 2026-09-08 Tue 04:00:45 PM, (NY) 2026-09-08 Tue 12:00:45 PM"
2,316.00,316.01,316.05,316.00,510.825,"(utc) 2026-09-08 Tue 04:01:45 PM, (NY) 2026-09-08 Tue 12:01:45 PM"
3,316.02,316.05,316.05,316.01,194.875,"(utc) 2026-09-08 Tue 04:02:45 PM, (NY) 2026-09-08 Tue 12:02:45 PM"
4,316.05,315.91,316.05,315.90,565.700,"(utc) 2026-09-08 Tue 04:03:45 PM, (NY) 2026-09-08 Tue 12:03:45 PM"
...,...,...,...,...,...,...
955,315.93,315.98,316.00,315.93,1245.650,"(utc) 2026-09-09 Wed 07:54:45 AM, (NY) 2026-09-09 Wed 03:54:45 AM"
956,315.98,315.96,316.03,315.94,1434.450,"(utc) 2026-09-09 Wed 07:55:45 AM, (NY) 2026-09-09 Wed 03:55:45 AM"
957,315.95,315.86,315.96,315.80,2482.625,"(utc) 2026-09-09 Wed 07:56:45 AM, (NY) 2026-09-09 Wed 03:56:45 AM"
958,315.85,315.98,315.98,315.85,2374.750,"(utc) 2026-09-09 Wed 07:57:45 AM, (NY) 2026-09-09 Wed 03:57:45 AM"


In [9]:
dataQuery(
    conid = 265598,
    bar = '30S',
    period = '480min',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-09-04 Fri 06:29:30 PM, (NY) 2026-09-04 Fri 02:29:30 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          28800s
barLength:           30


,o,c,h,l,v,t
0,320.92,320.87,320.92,320.87,194.100,"(utc) 2026-09-04 Fri 06:29:30 PM, (NY) 2026-09-04 Fri 02:29:30 PM"
1,320.88,321.01,321.01,320.86,292.975,"(utc) 2026-09-04 Fri 06:30:30 PM, (NY) 2026-09-04 Fri 02:30:30 PM"
2,321.02,320.99,321.03,320.97,183.375,"(utc) 2026-09-04 Fri 06:31:30 PM, (NY) 2026-09-04 Fri 02:31:30 PM"
3,320.99,321.01,321.03,320.99,235.450,"(utc) 2026-09-04 Fri 06:32:30 PM, (NY) 2026-09-04 Fri 02:32:30 PM"
4,321.00,321.06,321.07,320.98,251.400,"(utc) 2026-09-04 Fri 06:33:30 PM, (NY) 2026-09-04 Fri 02:33:30 PM"
...,...,...,...,...,...,...
955,315.96,315.85,315.98,315.70,2437.125,"(utc) 2026-09-12 Sat 09:24:30 PM, (NY) 2026-09-12 Sat 05:24:30 PM"
956,315.85,315.86,315.91,315.81,1697.850,"(utc) 2026-09-12 Sat 09:25:30 PM, (NY) 2026-09-12 Sat 05:25:30 PM"
957,315.87,315.93,315.95,315.82,2750.325,"(utc) 2026-09-12 Sat 09:26:30 PM, (NY) 2026-09-12 Sat 05:26:30 PM"
958,315.93,315.96,316.03,315.93,2680.100,"(utc) 2026-09-12 Sat 09:27:30 PM, (NY) 2026-09-12 Sat 05:27:30 PM"


From here on are larger bars, the rate limits are higher and the dates don't have to be within the last 6 months

In [10]:
dataQuery(
    conid = 265598,
    bar = '1min',
    period = '16h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-09-03 Thu 04:59:00 PM, (NY) 2026-09-03 Thu 12:59:00 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          57600s
barLength:           60


,o,c,h,l,v,t
0,329.02,328.94,329.06,328.88,883.375,"(utc) 2026-09-03 Thu 04:59:00 PM, (NY) 2026-09-03 Thu 12:59:00 PM"
1,328.95,328.99,329.02,328.90,1205.250,"(utc) 2026-09-03 Thu 05:00:00 PM, (NY) 2026-09-03 Thu 01:00:00 PM"
2,328.97,328.83,329.00,328.72,674.425,"(utc) 2026-09-03 Thu 05:01:00 PM, (NY) 2026-09-03 Thu 01:01:00 PM"
3,328.85,328.48,328.86,328.47,715.825,"(utc) 2026-09-03 Thu 05:02:00 PM, (NY) 2026-09-03 Thu 01:02:00 PM"
4,328.50,328.62,328.63,328.44,682.550,"(utc) 2026-09-03 Thu 05:03:00 PM, (NY) 2026-09-03 Thu 01:03:00 PM"
...,...,...,...,...,...,...
955,316.08,316.43,316.45,315.99,3346.625,"(utc) 2026-09-08 Tue 07:54:00 PM, (NY) 2026-09-08 Tue 03:54:00 PM"
956,316.45,316.18,316.85,316.14,4924.325,"(utc) 2026-09-08 Tue 07:55:00 PM, (NY) 2026-09-08 Tue 03:55:00 PM"
957,316.19,315.96,316.21,315.93,3475.375,"(utc) 2026-09-08 Tue 07:56:00 PM, (NY) 2026-09-08 Tue 03:56:00 PM"
958,315.96,315.86,315.98,315.70,4134.975,"(utc) 2026-09-08 Tue 07:57:00 PM, (NY) 2026-09-08 Tue 03:57:00 PM"


In [11]:
dataQuery(
    conid = 265598,
    bar = '2min',
    period = '32h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-09-01 Tue 01:58:00 PM, (NY) 2026-09-01 Tue 09:58:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          115200s
barLength:           120


,o,c,h,l,v,t
0,321.13,322.70,322.75,321.13,14776.225,"(utc) 2026-09-01 Tue 01:58:00 PM, (NY) 2026-09-01 Tue 09:58:00 AM"
1,322.70,322.97,323.39,322.58,17270.300,"(utc) 2026-09-01 Tue 02:00:00 PM, (NY) 2026-09-01 Tue 10:00:00 AM"
2,322.98,322.00,322.98,321.84,10650.125,"(utc) 2026-09-01 Tue 02:02:00 PM, (NY) 2026-09-01 Tue 10:02:00 AM"
3,322.02,322.21,322.63,321.47,8372.950,"(utc) 2026-09-01 Tue 02:04:00 PM, (NY) 2026-09-01 Tue 10:04:00 AM"
4,322.18,323.23,323.28,322.18,8255.925,"(utc) 2026-09-01 Tue 02:06:00 PM, (NY) 2026-09-01 Tue 10:06:00 AM"
...,...,...,...,...,...,...
955,316.11,316.25,316.27,316.03,3818.550,"(utc) 2026-09-08 Tue 07:48:00 PM, (NY) 2026-09-08 Tue 03:48:00 PM"
956,316.27,316.08,316.28,315.86,4871.300,"(utc) 2026-09-08 Tue 07:50:00 PM, (NY) 2026-09-08 Tue 03:50:00 PM"
957,316.07,316.08,316.34,316.00,4981.375,"(utc) 2026-09-08 Tue 07:52:00 PM, (NY) 2026-09-08 Tue 03:52:00 PM"
958,316.08,316.18,316.85,315.99,8270.950,"(utc) 2026-09-08 Tue 07:54:00 PM, (NY) 2026-09-08 Tue 03:54:00 PM"


In [12]:
dataQuery(
    conid = 265598,
    bar = '3min',
    period = '48h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-08-27 Thu 05:27:00 PM, (NY) 2026-08-27 Thu 01:27:00 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          172800s
barLength:           180


,o,c,h,l,v,t
0,315.12,315.27,315.27,315.11,1560.750,"(utc) 2026-08-27 Thu 05:27:00 PM, (NY) 2026-08-27 Thu 01:27:00 PM"
1,315.28,315.10,315.34,315.08,1713.950,"(utc) 2026-08-27 Thu 05:30:00 PM, (NY) 2026-08-27 Thu 01:30:00 PM"
2,315.08,314.84,315.17,314.82,1829.900,"(utc) 2026-08-27 Thu 05:33:00 PM, (NY) 2026-08-27 Thu 01:33:00 PM"
3,314.85,314.60,314.87,314.60,1479.300,"(utc) 2026-08-27 Thu 05:36:00 PM, (NY) 2026-08-27 Thu 01:36:00 PM"
4,314.59,314.72,314.73,314.49,1878.950,"(utc) 2026-08-27 Thu 05:39:00 PM, (NY) 2026-08-27 Thu 01:39:00 PM"
...,...,...,...,...,...,...
955,316.21,316.18,316.24,316.02,4142.375,"(utc) 2026-09-08 Tue 07:42:00 PM, (NY) 2026-09-08 Tue 03:42:00 PM"
956,316.20,316.12,316.24,315.91,3532.750,"(utc) 2026-09-08 Tue 07:45:00 PM, (NY) 2026-09-08 Tue 03:45:00 PM"
957,316.11,316.02,316.28,315.86,6784.650,"(utc) 2026-09-08 Tue 07:48:00 PM, (NY) 2026-09-08 Tue 03:48:00 PM"
958,316.02,316.08,316.34,315.92,6886.575,"(utc) 2026-09-08 Tue 07:51:00 PM, (NY) 2026-09-08 Tue 03:51:00 PM"


In [2]:
dataQuery(
    conid = 265598,
    bar = '5min',
    period = '80h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-08-20 Thu 05:55:00 PM, (NY) 2026-08-20 Thu 01:55:00 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          288000s
barLength:           300


,o,c,h,l,v,t
0,316.56,316.32,316.62,316.27,2105.475,"(utc) 2026-08-20 Thu 05:55:00 PM, (NY) 2026-08-20 Thu 01:55:00 PM"
1,316.33,316.11,316.33,316.00,5780.975,"(utc) 2026-08-20 Thu 06:00:00 PM, (NY) 2026-08-20 Thu 02:00:00 PM"
2,316.11,315.31,316.13,315.28,6464.550,"(utc) 2026-08-20 Thu 06:05:00 PM, (NY) 2026-08-20 Thu 02:05:00 PM"
3,315.32,315.45,315.58,315.08,7832.925,"(utc) 2026-08-20 Thu 06:10:00 PM, (NY) 2026-08-20 Thu 02:10:00 PM"
4,315.47,315.61,315.65,315.29,4638.475,"(utc) 2026-08-20 Thu 06:15:00 PM, (NY) 2026-08-20 Thu 02:15:00 PM"
...,...,...,...,...,...,...
955,315.80,316.05,316.24,315.52,5932.350,"(utc) 2026-09-08 Tue 07:30:00 PM, (NY) 2026-09-08 Tue 03:30:00 PM"
956,316.05,316.30,316.32,315.96,5270.400,"(utc) 2026-09-08 Tue 07:35:00 PM, (NY) 2026-09-08 Tue 03:35:00 PM"
957,316.27,316.18,316.36,316.02,6582.475,"(utc) 2026-09-08 Tue 07:40:00 PM, (NY) 2026-09-08 Tue 03:40:00 PM"
958,316.20,316.25,316.27,315.91,7351.300,"(utc) 2026-09-08 Tue 07:45:00 PM, (NY) 2026-09-08 Tue 03:45:00 PM"


In [3]:
# 25 days also works but this leaves a bit more breathing room
dataQuery(
    conid = 265598,
    bar = '10min',
    period = '160h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-08-04 Tue 03:50:00 PM, (NY) 2026-08-04 Tue 11:50:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          576000s
barLength:           600


,o,c,h,l,v,t
0,306.38,306.34,306.67,306.13,11953.400,"(utc) 2026-08-04 Tue 03:50:00 PM, (NY) 2026-08-04 Tue 11:50:00 AM"
1,306.34,305.93,306.45,305.67,13626.175,"(utc) 2026-08-04 Tue 04:00:00 PM, (NY) 2026-08-04 Tue 12:00:00 PM"
2,305.92,305.87,306.40,305.83,11539.775,"(utc) 2026-08-04 Tue 04:10:00 PM, (NY) 2026-08-04 Tue 12:10:00 PM"
3,305.87,306.50,306.72,305.82,14295.575,"(utc) 2026-08-04 Tue 04:20:00 PM, (NY) 2026-08-04 Tue 12:20:00 PM"
4,306.50,306.74,306.96,306.34,15080.325,"(utc) 2026-08-04 Tue 04:30:00 PM, (NY) 2026-08-04 Tue 12:30:00 PM"
...,...,...,...,...,...,...
955,315.52,316.27,316.33,315.35,10473.375,"(utc) 2026-09-08 Tue 07:00:00 PM, (NY) 2026-09-08 Tue 03:00:00 PM"
956,316.27,315.79,316.53,315.60,13528.750,"(utc) 2026-09-08 Tue 07:10:00 PM, (NY) 2026-09-08 Tue 03:10:00 PM"
957,315.79,315.80,316.05,315.52,10203.475,"(utc) 2026-09-08 Tue 07:20:00 PM, (NY) 2026-09-08 Tue 03:20:00 PM"
958,315.80,316.30,316.32,315.52,11202.750,"(utc) 2026-09-08 Tue 07:30:00 PM, (NY) 2026-09-08 Tue 03:30:00 PM"


In [5]:
dataQuery(
    conid = 265598,
    bar = '15min',
    period = '240h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-07-17 Fri 01:45:00 PM, (NY) 2026-07-17 Fri 09:45:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          864000s
barLength:           900


,o,c,h,l,v,t
0,333.89,330.92,334.40,330.89,61243.800,"(utc) 2026-07-17 Fri 01:45:00 PM, (NY) 2026-07-17 Fri 09:45:00 AM"
1,330.94,330.51,330.99,329.00,52059.475,"(utc) 2026-07-17 Fri 02:00:00 PM, (NY) 2026-07-17 Fri 10:00:00 AM"
2,330.48,331.84,331.85,329.84,33606.950,"(utc) 2026-07-17 Fri 02:15:00 PM, (NY) 2026-07-17 Fri 10:15:00 AM"
3,331.88,333.07,333.97,331.85,49611.425,"(utc) 2026-07-17 Fri 02:30:00 PM, (NY) 2026-07-17 Fri 10:30:00 AM"
4,333.06,333.17,333.59,331.91,29653.375,"(utc) 2026-07-17 Fri 02:45:00 PM, (NY) 2026-07-17 Fri 10:45:00 AM"
...,...,...,...,...,...,...
955,316.40,315.47,316.54,315.38,19099.325,"(utc) 2026-09-08 Tue 06:30:00 PM, (NY) 2026-09-08 Tue 02:30:00 PM"
956,315.47,315.53,315.98,314.90,22713.725,"(utc) 2026-09-08 Tue 06:45:00 PM, (NY) 2026-09-08 Tue 02:45:00 PM"
957,315.52,316.09,316.53,315.35,18593.300,"(utc) 2026-09-08 Tue 07:00:00 PM, (NY) 2026-09-08 Tue 03:00:00 PM"
958,316.09,315.80,316.09,315.52,15612.300,"(utc) 2026-09-08 Tue 07:15:00 PM, (NY) 2026-09-08 Tue 03:15:00 PM"


In [18]:
dataQuery(
    conid = 265598,
    bar = '20min',
    period = '320h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-06-29 Mon 06:00:00 PM, (NY) 2026-06-29 Mon 02:00:00 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          1152000s
barLength:           1200


,o,c,h,l,v,t
0,282.03,281.42,282.05,281.00,22247.400,"(utc) 2026-06-29 Mon 06:00:00 PM, (NY) 2026-06-29 Mon 02:00:00 PM"
1,281.43,281.94,282.35,280.89,35068.800,"(utc) 2026-06-29 Mon 06:20:00 PM, (NY) 2026-06-29 Mon 02:20:00 PM"
2,281.95,282.07,282.28,281.45,35940.525,"(utc) 2026-06-29 Mon 06:40:00 PM, (NY) 2026-06-29 Mon 02:40:00 PM"
3,282.08,282.60,283.08,281.93,40069.350,"(utc) 2026-06-29 Mon 07:00:00 PM, (NY) 2026-06-29 Mon 03:00:00 PM"
4,282.60,282.30,282.89,281.77,42356.125,"(utc) 2026-06-29 Mon 07:20:00 PM, (NY) 2026-06-29 Mon 03:20:00 PM"
...,...,...,...,...,...,...
980,316.03,316.28,316.52,315.97,13463.875,"(utc) 2026-09-08 Tue 06:00:00 PM, (NY) 2026-09-08 Tue 02:00:00 PM"
981,316.27,315.86,316.74,315.72,23785.500,"(utc) 2026-09-08 Tue 06:20:00 PM, (NY) 2026-09-08 Tue 02:20:00 PM"
982,315.85,315.53,315.98,314.90,28988.425,"(utc) 2026-09-08 Tue 06:40:00 PM, (NY) 2026-09-08 Tue 02:40:00 PM"
983,315.52,315.79,316.53,315.35,24002.125,"(utc) 2026-09-08 Tue 07:00:00 PM, (NY) 2026-09-08 Tue 03:00:00 PM"


In [8]:
dataQuery(
    conid = 265598,
    bar = '30min',
    period = '480h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-05-22 Fri 02:00:00 PM, (NY) 2026-05-22 Fri 10:00:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          1728000s
barLength:           1800


,o,c,h,l,v,t
0,308.69,308.64,309.27,308.00,69692.450,"(utc) 2026-05-22 Fri 02:00:00 PM, (NY) 2026-05-22 Fri 10:00:00 AM"
1,308.63,310.33,310.48,308.62,65523.000,"(utc) 2026-05-22 Fri 02:30:00 PM, (NY) 2026-05-22 Fri 10:30:00 AM"
2,310.33,311.00,311.40,310.17,62709.050,"(utc) 2026-05-22 Fri 03:00:00 PM, (NY) 2026-05-22 Fri 11:00:00 AM"
3,311.00,309.98,311.04,309.85,39846.975,"(utc) 2026-05-22 Fri 03:30:00 PM, (NY) 2026-05-22 Fri 11:30:00 AM"
4,309.98,309.92,310.53,309.72,35065.625,"(utc) 2026-05-22 Fri 04:00:00 PM, (NY) 2026-05-22 Fri 12:00:00 PM"
...,...,...,...,...,...,...
955,315.99,316.27,316.56,315.79,20295.850,"(utc) 2026-09-08 Tue 05:00:00 PM, (NY) 2026-09-08 Tue 01:00:00 PM"
956,316.27,316.03,316.33,315.94,22035.600,"(utc) 2026-09-08 Tue 05:30:00 PM, (NY) 2026-09-08 Tue 01:30:00 PM"
957,316.03,316.39,316.74,315.95,24424.750,"(utc) 2026-09-08 Tue 06:00:00 PM, (NY) 2026-09-08 Tue 02:00:00 PM"
958,316.40,315.53,316.54,314.90,41813.050,"(utc) 2026-09-08 Tue 06:30:00 PM, (NY) 2026-09-08 Tue 02:30:00 PM"


In [13]:
dataQuery(
    conid = 265598,
    bar = '1h',
    period = '880h',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-02-24 Tue 05:00:00 PM, (NY) 2026-02-24 Tue 12:00:00 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          3168000s
barLength:           3600


,o,c,h,l,v,t
0,272.46,272.29,272.87,271.82,53427.425,"(utc) 2026-02-24 Tue 05:00:00 PM, (NY) 2026-02-24 Tue 12:00:00 PM"
1,272.31,271.96,272.69,271.37,41209.575,"(utc) 2026-02-24 Tue 06:00:00 PM, (NY) 2026-02-24 Tue 01:00:00 PM"
2,271.96,272.63,272.75,271.74,41321.950,"(utc) 2026-02-24 Tue 07:00:00 PM, (NY) 2026-02-24 Tue 02:00:00 PM"
3,272.65,272.17,272.67,271.38,88681.000,"(utc) 2026-02-24 Tue 08:00:00 PM, (NY) 2026-02-24 Tue 03:00:00 PM"
4,271.73,273.93,274.00,271.05,85317.725,"(utc) 2026-02-25 Wed 02:30:00 PM, (NY) 2026-02-25 Wed 09:30:00 AM"
...,...,...,...,...,...,...
943,316.92,315.97,317.27,315.27,94758.875,"(utc) 2026-09-08 Tue 02:00:00 PM, (NY) 2026-09-08 Tue 10:00:00 AM"
944,316.00,316.08,316.82,315.71,65504.000,"(utc) 2026-09-08 Tue 03:00:00 PM, (NY) 2026-09-08 Tue 11:00:00 AM"
945,316.06,315.99,316.27,315.46,49897.175,"(utc) 2026-09-08 Tue 04:00:00 PM, (NY) 2026-09-08 Tue 12:00:00 PM"
946,315.99,316.03,316.56,315.79,42331.450,"(utc) 2026-09-08 Tue 05:00:00 PM, (NY) 2026-09-08 Tue 01:00:00 PM"


In [15]:
# While we could fetch more in this query, we start hitting the 10 sec limit 
# You can push it higher, 35w was around 8 secs in my testing. Using months wasn't any faster
dataQuery(
    conid = 265598,
    bar = '2h',
    period = '35w',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          35w
barLength:           7200


,o,c,h,l,v,t
0,263.20,261.99,263.68,261.21,222063.825,"(utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM"
1,261.99,262.31,262.70,260.90,141107.150,"(utc) 2026-01-07 Wed 04:00:00 PM, (NY) 2026-01-07 Wed 11:00:00 AM"
2,262.31,261.84,262.52,261.08,129753.075,"(utc) 2026-01-07 Wed 06:00:00 PM, (NY) 2026-01-07 Wed 01:00:00 PM"
3,261.85,260.36,261.98,259.81,153620.200,"(utc) 2026-01-07 Wed 08:00:00 PM, (NY) 2026-01-07 Wed 03:00:00 PM"
4,257.06,256.68,257.78,255.70,314023.825,"(utc) 2026-01-08 Thu 02:30:00 PM, (NY) 2026-01-08 Thu 09:30:00 AM"
...,...,...,...,...,...,...
666,321.03,321.02,321.90,320.41,112248.800,"(utc) 2026-09-04 Fri 04:00:00 PM, (NY) 2026-09-04 Fri 12:00:00 PM"
667,321.02,319.99,322.10,319.65,164263.025,"(utc) 2026-09-04 Fri 06:00:00 PM, (NY) 2026-09-04 Fri 02:00:00 PM"
668,317.10,316.90,320.70,315.70,95295.400,"(utc) 2026-09-08 Tue 01:30:00 PM, (NY) 2026-09-08 Tue 09:30:00 AM"
669,316.92,316.08,317.27,315.27,160262.875,"(utc) 2026-09-08 Tue 02:00:00 PM, (NY) 2026-09-08 Tue 10:00:00 AM"


In [30]:
dataQuery(
    conid = 265598,
    bar = '3h',
    period = '35w',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          35w
barLength:           10800


,o,c,h,l,v,t
0,263.20,261.40,263.68,261.23,116038.150,"(utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM"
1,261.44,262.31,262.70,260.90,247132.825,"(utc) 2026-01-07 Wed 03:00:00 PM, (NY) 2026-01-07 Wed 10:00:00 AM"
2,262.31,260.36,262.52,259.81,283373.275,"(utc) 2026-01-07 Wed 06:00:00 PM, (NY) 2026-01-07 Wed 01:00:00 PM"
3,257.06,256.29,257.78,255.85,166584.075,"(utc) 2026-01-08 Thu 02:30:00 PM, (NY) 2026-01-08 Thu 09:30:00 AM"
4,256.28,257.39,258.60,255.70,330381.375,"(utc) 2026-01-08 Thu 03:00:00 PM, (NY) 2026-01-08 Thu 10:00:00 AM"
...,...,...,...,...,...,...
498,328.29,318.45,328.93,318.30,255092.125,"(utc) 2026-09-04 Fri 01:30:00 PM, (NY) 2026-09-04 Fri 09:30:00 AM"
499,318.44,321.02,321.90,317.86,224381.775,"(utc) 2026-09-04 Fri 03:00:00 PM, (NY) 2026-09-04 Fri 11:00:00 AM"
500,321.02,319.99,322.10,319.65,164263.025,"(utc) 2026-09-04 Fri 06:00:00 PM, (NY) 2026-09-04 Fri 02:00:00 PM"
501,317.10,315.97,320.70,315.27,190054.275,"(utc) 2026-09-08 Tue 01:30:00 PM, (NY) 2026-09-08 Tue 09:30:00 AM"


In [32]:
dataQuery(
    conid = 265598,
    bar = '4h',
    period = '35w',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          35w
barLength:           14400


,o,c,h,l,v,t
0,263.20,261.99,263.68,261.21,222063.825,"(utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM"
1,261.99,261.84,262.70,260.90,270860.225,"(utc) 2026-01-07 Wed 04:00:00 PM, (NY) 2026-01-07 Wed 11:00:00 AM"
2,261.85,260.36,261.98,259.81,153620.200,"(utc) 2026-01-07 Wed 08:00:00 PM, (NY) 2026-01-07 Wed 03:00:00 PM"
3,257.06,256.68,257.78,255.70,314023.825,"(utc) 2026-01-08 Thu 02:30:00 PM, (NY) 2026-01-08 Thu 09:30:00 AM"
4,256.69,257.22,258.60,255.97,295392.050,"(utc) 2026-01-08 Thu 04:00:00 PM, (NY) 2026-01-08 Thu 11:00:00 AM"
...,...,...,...,...,...,...
371,324.87,329.77,330.81,324.11,297828.125,"(utc) 2026-09-03 Thu 01:30:00 PM, (NY) 2026-09-03 Thu 09:30:00 AM"
372,329.77,328.22,330.57,326.51,277135.125,"(utc) 2026-09-03 Thu 04:00:00 PM, (NY) 2026-09-03 Thu 12:00:00 PM"
373,328.29,321.02,328.93,317.86,367225.100,"(utc) 2026-09-04 Fri 01:30:00 PM, (NY) 2026-09-04 Fri 09:30:00 AM"
374,321.03,319.99,322.10,319.65,276511.825,"(utc) 2026-09-04 Fri 04:00:00 PM, (NY) 2026-09-04 Fri 12:00:00 PM"


In [33]:
dataQuery(
    conid = 265598,
    bar = '8h',
    period = '35w',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          35w
barLength:           28800


,o,c,h,l,v,t
0,263.20,261.99,263.68,261.21,222063.825,"(utc) 2026-01-07 Wed 02:30:00 PM, (NY) 2026-01-07 Wed 09:30:00 AM"
1,261.99,260.36,262.70,259.81,424480.425,"(utc) 2026-01-07 Wed 04:00:00 PM, (NY) 2026-01-07 Wed 11:00:00 AM"
2,257.06,256.68,257.78,255.70,314023.825,"(utc) 2026-01-08 Thu 02:30:00 PM, (NY) 2026-01-08 Thu 09:30:00 AM"
3,256.69,259.08,259.29,255.97,427570.575,"(utc) 2026-01-08 Thu 04:00:00 PM, (NY) 2026-01-08 Thu 11:00:00 AM"
4,259.07,257.78,260.00,256.22,213852.900,"(utc) 2026-01-09 Fri 02:30:00 PM, (NY) 2026-01-09 Fri 09:30:00 AM"
...,...,...,...,...,...,...
330,324.87,329.77,330.81,324.11,297828.125,"(utc) 2026-09-03 Thu 01:30:00 PM, (NY) 2026-09-03 Thu 09:30:00 AM"
331,329.77,328.22,330.57,326.51,277135.125,"(utc) 2026-09-03 Thu 04:00:00 PM, (NY) 2026-09-03 Thu 12:00:00 PM"
332,328.29,321.02,328.93,317.86,367225.100,"(utc) 2026-09-04 Fri 01:30:00 PM, (NY) 2026-09-04 Fri 09:30:00 AM"
333,321.03,319.99,322.10,319.65,276511.825,"(utc) 2026-09-04 Fri 04:00:00 PM, (NY) 2026-09-04 Fri 12:00:00 PM"


In [38]:
dataQuery(
    conid = 265598,
    bar = '1d',
    period = '44m',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2023-01-24 Tue 02:30:00 PM, (NY) 2023-01-24 Tue 09:30:00 AM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          44m
barLength:           86400


,o,c,h,l,v,t
0,140.95,141.86,142.43,138.81,1236445.675,"(utc) 2023-01-25 Wed 02:30:00 PM, (NY) 2023-01-25 Wed 09:30:00 AM"
1,143.15,143.96,144.25,141.90,998448.600,"(utc) 2023-01-26 Thu 02:30:00 PM, (NY) 2023-01-26 Thu 09:30:00 AM"
2,143.18,145.93,147.23,143.08,1262177.950,"(utc) 2023-01-27 Fri 02:30:00 PM, (NY) 2023-01-27 Fri 09:30:00 AM"
3,144.96,143.00,145.55,142.85,1145039.500,"(utc) 2023-01-30 Mon 02:30:00 PM, (NY) 2023-01-30 Mon 09:30:00 AM"
4,142.69,144.29,144.34,142.28,943963.575,"(utc) 2023-01-31 Tue 02:30:00 PM, (NY) 2023-01-31 Tue 09:30:00 AM"
...,...,...,...,...,...,...
902,319.69,316.85,321.24,313.04,503727.875,"(utc) 2026-08-31 Mon 01:30:00 PM, (NY) 2026-08-31 Mon 09:30:00 AM"
903,317.00,325.13,327.30,314.73,906731.950,"(utc) 2026-09-01 Tue 01:30:00 PM, (NY) 2026-09-01 Tue 09:30:00 AM"
904,326.76,324.96,328.40,323.53,534102.150,"(utc) 2026-09-02 Wed 01:30:00 PM, (NY) 2026-09-02 Wed 09:30:00 AM"
905,324.87,328.21,330.81,324.11,574963.200,"(utc) 2026-09-03 Thu 01:30:00 PM, (NY) 2026-09-03 Thu 09:30:00 AM"


In [44]:
dataQuery(
    conid = 265598,
    bar = '1w',
    period = '210m',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

startTime:           (utc) 2009-01-05 Mon 12:00:00 AM, (NY) 2009-01-04 Sun 07:00:00 PM
chartPanStartTime:   (utc) 2026-09-08 Tue 08:00:00 PM, (NY) 2026-09-08 Tue 04:00:00 PM
timePeriod:          210m
barLength:           604800


,o,c,h,l,v,t
0,3.33,3.24,3.47,3.22,1.048163e+08,"(utc) 2009-01-05 Mon 12:00:00 AM, (NY) 2009-01-04 Sun 07:00:00 PM"
1,3.23,2.94,3.25,2.86,1.197900e+08,"(utc) 2009-01-12 Mon 12:00:00 AM, (NY) 2009-01-11 Sun 07:00:00 PM"
2,2.92,3.16,3.21,2.79,9.052043e+07,"(utc) 2009-01-20 Tue 12:00:00 AM, (NY) 2009-01-19 Mon 07:00:00 PM"
3,3.17,3.22,3.39,3.15,7.882350e+07,"(utc) 2009-01-26 Mon 12:00:00 AM, (NY) 2009-01-25 Sun 07:00:00 PM"
4,3.18,3.56,3.57,3.18,7.942284e+07,"(utc) 2009-02-02 Mon 12:00:00 AM, (NY) 2009-02-01 Sun 07:00:00 PM"
...,...,...,...,...,...,...
917,309.73,313.33,316.29,301.32,4.044865e+06,"(utc) 2026-08-03 Mon 12:00:00 AM, (NY) 2026-08-02 Sun 08:00:00 PM"
918,306.85,305.93,309.97,300.57,2.791775e+06,"(utc) 2026-08-10 Mon 12:00:00 AM, (NY) 2026-08-09 Sun 08:00:00 PM"
919,306.00,309.35,320.27,302.96,2.967395e+06,"(utc) 2026-08-17 Mon 12:00:00 AM, (NY) 2026-08-16 Sun 08:00:00 PM"
920,311.53,319.70,322.37,308.21,2.244497e+06,"(utc) 2026-08-24 Mon 12:00:00 AM, (NY) 2026-08-23 Sun 08:00:00 PM"


In [51]:
dataQuery(
    conid = 265598,
    bar = '1m',
    period = '30y',
    startTime = datetime(2026, 9, 8, 20, 0, 0),
    outsideRth = False
)

KeyError: 'startTime'